# Categorical Variables

Columns with a **limited set of fixed values** — like departments, gender, grades.

- **Nominal** — no natural order (dept, color, city)
- **Ordinal** — has natural order (grade, size, rating)

In [1]:
import pandas as pd

df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Charlie", "David"],
    "dept":  ["HR", "IT", "HR", "Finance"],
    "grade": ["A", "B", "A", "C"],
    "score": [88, 92, 78, 85]
})
df

,name,dept,grade,score
0,Alice,HR,A,88
1,Bob,IT,B,92
2,Charlie,HR,A,78
3,David,Finance,C,85


## 1. Identify Categorical Columns

In [2]:
df.dtypes
# name     object  ← likely categorical
# dept     object
# grade    object
# score    int64   ← numeric

name     object
dept     object
grade    object
score     int64
dtype: object

In [3]:
# select only object (string) columns — likely categorical
df.select_dtypes(include="object").columns

Index(['name', 'dept', 'grade'], dtype='object')

## 2. Convert to `category` dtype

In [6]:
df["dept"] = df["dept"].astype("category")

print(df["dept"].cat.categories)   # Index(['Finance', 'HR', 'IT'])
print(df["dept"].cat.codes)        # integer code assigned to each value
# Benefits: less memory, faster groupby/sort
df.dtypes

Index(['Finance', 'HR', 'IT'], dtype='object')
0    1
1    2
2    1
3    0
dtype: int8


name       object
dept     category
grade      object
score       int64
dtype: object

In [24]:
df["dept"] = df["dept"].astype("category")
print(df["dept"].cat.categories)  
try:
    df["dept"]  = df["dept"].cat.remove_categories("Marketing") 
except AttributeError as e:                                                                                                                                                                  
    print("Column is not category dtype")                                                                                                                                               
except ValueError as e:                       
    print("Category not found")        

Index(['Finance', 'HR', 'IT'], dtype='object')
Category not found


In [25]:
# what if someone added a row with a diff dept                                                                                                                                            
                                               
# ● Great question — Pandas will either reject it or set it to NaN, depending on how you add the row.                                                                                       
                                                                                                                                                                              

print(df["dept"].cat.categories)   # Index(['Finance', 'HR', 'IT'])                                                                                                                     
                                                                                                                                                                                          
#   ---                                                                                                                                                                                     
#   Adding a new value not in categories:                                                                                                                                                   
                                                                                                                                                                                          
  # Using loc — sets to NaN, no error                       
df.loc[4] = ["Eve", "Marketing", "A", 90]                                                                                                                                               
df["dept"]                                                                                                                                                                              
  # 0         HR                                                                                                                                                                          
  # 1         IT                                                                                                                                                                          
  # 2         HR                                            
  # 3    Finance
  # 4        NaN   ← "Marketing" silently dropped!                                                                                                                                        
   
#   ---                                                                                                                                                                                     
#   Using concat — reverts column back to object dtype:       
                                                                                                                                                                                          
new_row = pd.DataFrame([{"name": "Eve", "dept": "Marketing", "grade": "A", "score": 90}])
df2 = pd.concat([df, new_row])                                                                                                                                                          
df2["dept"].dtype    # object  ← category is gone!
                                                                                                                                                                                          
#   ---                                                       
#   The correct way — add the new category first:                                                                                                                                           
                                                                                                                                                                                          
df["dept"] = df["dept"].cat.add_categories("Marketing")
df.loc[4] = ["Eve", "Marketing", "A", 90]                                                                                                                                               
df["dept"]                                                                                                                                                                              
  # 0         HR
  # 1         IT                                                                                                                                                                          
  # 2         HR                                            
  # 3    Finance
  # 4  Marketing   ✓

#   ---
#   This is the key difference from Java enums:
                                                                                                                                                                                          
#   // Java — compile error, can't add new enum value at runtime
#   Grade g = Grade.DISTINCTION;  // error: DISTINCTION doesn't exist                                                                                                                       
                                                                                                                                                                                          
  # Python — silently fails or loses dtype, no compile-time safety                                                                                                                        
                                                                                                                                                                                          
#   ---                                                                                                                                                                                     
#   Useful cat methods for managing categories:                                                                                                                                             
                                                                                                                                                                                          
df["dept"].cat.add_categories("Marketing")     # add one  
df["dept"].cat.remove_categories("Finance")    # remove one                                                                                                                             
df["dept"].cat.rename_categories({"HR": "HumanResources"})
df["dept"].cat.set_categories(["HR", "IT"])    # replace entire set  

Index(['Finance', 'HR', 'IT'], dtype='object')


TypeError: Cannot setitem on a Categorical with a new category (Marketing), set the categories first

## 3. Ordinal Categories (order matters)

In [27]:
from pandas.api.types import CategoricalDtype

grade_order = CategoricalDtype(categories=["C", "B", "A"], ordered=True)
df["grade"] = df["grade"].astype(grade_order)

print(df["grade"].cat.ordered)     # True
print(df[df["grade"] > "B"])       # only grade A rows
print("---")
print(df.sort_values("grade"))     # sorts C → B → A

True
      name dept grade  score
0    Alice   HR     A     88
2  Charlie   HR     A     78
4      Eve  NaN     A     90
---
      name     dept grade  score
3    David  Finance     C     85
1      Bob       IT     B     92
0    Alice       HR     A     88
2  Charlie       HR     A     78
4      Eve      NaN     A     90


## 4. Encoding for ML

In [ ]:
# Label encoding — ordinal, one column (A=2, B=1, C=0)
df["grade_code"] = df["grade"].cat.codes
print(df[["grade", "grade_code"]])

# One-hot encoding — nominal, creates a column per category
pd.get_dummies(df, columns=["dept"])

  grade  grade_code
0     A           2
1     B           1
2     A           2
3     C           0
4     A           2


,name,grade,score,grade_code,dept_Finance,dept_HR,dept_IT
0,Alice,A,88,2,False,True,False
1,Bob,B,92,1,False,False,True
2,Charlie,A,78,2,False,True,False
3,David,C,85,0,True,False,False
4,Eve,A,90,2,False,False,False


## 5. Analyze Categorical Columns

In [29]:
print(df["dept"].value_counts())            # frequency per category
print(df.groupby("dept")["score"].mean())   # mean score per dept
print(df["dept"].nunique())                 # number of unique categories

dept
HR         2
Finance    1
IT         1
Name: count, dtype: int64
dept
Finance    85.0
HR         83.0
IT         92.0
Name: score, dtype: float64
3


/tmp/ipykernel_1823/3989308474.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("dept")["score"].mean())   # mean score per dept


## 6. Months of Year — Ordered Categorical Example

In [ ]:
import pandas as pd
from pandas.api.types import CategoricalDtype

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

month_type = CategoricalDtype(categories=months, ordered=True)

df = pd.DataFrame({
    "month": ["Mar", "Jan", "Dec", "Jun"],
    "sales": [300, 150, 500, 420]
})

df["month"] = df["month"].astype(month_type)
print(df.dtypes)
print(df)

In [ ]:
# Sort by calendar order, not alphabetical
df.sort_values("month")

In [ ]:
# Filter months after Jun — works because ordered=True
df[df["month"] > "Jun"]

## Without ordering — sorting goes alphabetical (wrong)

In [ ]:
df2 = pd.DataFrame({
    "month": ["Mar", "Jan", "Dec", "Jun"],
    "sales": [300, 150, 500, 420]
})
df2["month"] = df2["month"].astype("category")
print("Alphabetical order:", df2["month"].cat.categories.tolist())
df2.sort_values("month")  # wrong — Dec, Jan, Jun, Mar

## Datetime approach — when you have full dates

In [35]:
df3 = pd.DataFrame({
    "date": pd.to_datetime(["2024-03-01", "2024-01-01", "2024-12-01", "2024-06-01"]),
    "sales": [300, 150, 500, 420]
})

df3["month_name"] = df3["date"].dt.month_name()   # 'March', 'January'...
df3["month_num"]  = df3["date"].dt.month          # 1, 2, 3...
df3["month_abbr"] = df3["date"].dt.strftime("%b") # 'Mar', 'Jan'...
df3

df4 = pd.DataFrame({
    "date":["2024-03-01", "2024-01-01", "2024-12-01", "2024-06-01"],
    "sales": [300, 150, 500, 420]
})
# df4["date"] = df4["date"].astype("date")
df4["date"] = pd.to_datetime(df4["date"])
df4.dtypes
df3.dtypes

date          datetime64[ns]
sales                  int64
month_name            object
month_num              int32
month_abbr            object
dtype: object

In [ ]:
                                                                                                                                                                                        
df = pd.DataFrame({"color": ["R", "G", "B", "R", "G"]})
df.select_dtypes(include="object")
                                                                                                                                                                                        
# Without drop_first — 3 columns, redundant               
pd.get_dummies(df, columns=["color"])                                                                                                                                                   
#    color_B  color_G  color_R                                                                                                                                                          
# 0        0        0        1
# 1        0        1        0                                                                                                                                                          
# 2        1        0        0                                                                                                                                                          

# With drop_first — 2 columns, no redundancy                                                                                                                                            
pd.get_dummies(df, columns=["color"], drop_first=True)    
#    color_G  color_R
# 0        0        1                                                                                                                                                                   
# 1        1        0
# 2        0        0   ← both 0 means Blue                                                                                                                                             
                                                                                                                                                                                        
# ---
# The math behind it:                                                                                                                                                                     
                                                        
# color_B = 1 - color_R - color_G   ← always derivable
                                                                                                                                                                                        
# So color_B carries zero new information — keeping it causes the dummy variable trap.                                                                                                    
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# General rule:                                             

# ┌────────────┬────────────────┐
# │ Categories │ Columns needed │
# ├────────────┼────────────────┤
# │ 2 (Yes/No) │ 1              │
# ├────────────┼────────────────┤
# │ 3 (R/G/B)  │ 2              │                                                                                                                                                         
# ├────────────┼────────────────┤
# │ N          │ N-1            │                                                                                                                                                         
# └────────────┴────────────────┘                                                                                                                                                         

# ---                                                                                                                                                                                     
# When to drop vs keep:                                     
                    
# ┌────────────────────────────────┬───────────────────────────────────┐
# │             Model              │            Drop first?            │                                                                                                                  
# ├────────────────────────────────┼───────────────────────────────────┤
# │ Linear / Logistic Regression   │ yes — multicollinearity breaks it │                                                                                                                  
# ├────────────────────────────────┼───────────────────────────────────┤
# │ Decision Trees / Random Forest │ no — trees don't care             │                                                                                                                  
# ├────────────────────────────────┼───────────────────────────────────┤                                                                                                                  
# │ Neural Networks                │ no — handle redundancy fine       │                                                                                                                  
# └────────────────────────────────┴───────────────────────────────────┘               

In [ ]:
                                                                                                                                                                                  
df = pd.DataFrame({
    "name":  ["Alice", "Bob"],                                                                                                                                                          
    "dept":  ["HR", "IT"],                                                                                                                                                              
    "score": [88, 92],    
    "age":   [25, 30]                                                                                                                                                                   
})                                                        

df.select_dtypes(include="object")
#     name dept                                                                                                                                                                         
# 0  Alice   HR
# 1    Bob   IT                                                                                                                                                                         
                                                        

# To get just the column names — add .columns:
                                                                                                                                                                                        
df.select_dtypes(include="object").columns
# Index(['name', 'dept'])                                                                                                                                                               
                                                                                                                                                                                        
df.select_dtypes(include="object").columns.tolist()
# ['name', 'dept']                                                                                                                                                                      
                                                        
# ---
# Other useful variants:

df.select_dtypes(include="number")       # all numeric columns (int + float)
df.select_dtypes(include="float")        # float only                                                                                                                                   
df.select_dtypes(include="int")          # int only                                                                                                                                     
df.select_dtypes(include="category")     # category only                                                                                                                                
df.select_dtypes(include="datetime")     # datetime only                                                                                                                                
                                                        
df.select_dtypes(exclude="object")       # everything except strings                                                                                                                    
df.select_dtypes(exclude=["object", "category"])  # exclude multiple
                                                                                                                                                                                        
                                                    
# Common use case — apply operation to all string columns at once:
                                                                                                                                                                                        
# convert all object columns to category in one line
for col in df.select_dtypes(include="object").columns:                                                                                                                                  
    df[col] = df[col].astype("category")    

In [ ]:
df3["date"] = pd.to_datetime(["2024-01-01", "2024-06-01"])                                                                                                                              
                                                                                                                                                                                        
df3["date"].dt.month_name()       # 'January',  'June'      — full name                                                                                                                 
df3["date"].dt.strftime("%b")     # 'Jan',      'Jun'       — abbreviated                                                                                                               
df3["date"].dt.strftime("%B")     # 'January',  'June'      — full name                                                                                                                 
                                                                                                                                                                                        
# So month_name() == strftime("%B") — both give full name.                                                                                                                                
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# Key differences:                                                                                                                                                                        
                                                        
# ┌───────────────┬───────────────┬────────────────┬────────────────┐
# │               │ month_name()  │ strftime("%b") │ strftime("%B") │                                                                                                                     
# ├───────────────┼───────────────┼────────────────┼────────────────┤
# │ Output        │ January       │ Jan            │ January        │                                                                                                                     
# ├───────────────┼───────────────┼────────────────┼────────────────┤
# │ Returns       │ Pandas Series │ Pandas Series  │ Pandas Series  │                                                                                                                     
# ├───────────────┼───────────────┼────────────────┼────────────────┤
# │ Locale aware  │ yes           │ yes            │ yes            │                                                                                                                     
# ├───────────────┼───────────────┼────────────────┼────────────────┤                                                                                                                     
# │ Pandas native │ yes           │ via Python     │ via Python     │
# └───────────────┴───────────────┴────────────────┴────────────────┘                                                                                                                     
                                                        
# ---
# strftime can do much more — full date formatting:
                                                                                                                                                                                        
df3["date"].dt.strftime("%b")          # Jan
df3["date"].dt.strftime("%B")          # January                                                                                                                                        
df3["date"].dt.strftime("%b %Y")       # Jan 2024         
df3["date"].dt.strftime("%d-%m-%Y")    # 01-01-2024                                                                                                                                     
df3["date"].dt.strftime("%Y/%m/%d")    # 2024/01/01                                                                                                                                     
df3["date"].dt.strftime("%A")          # Monday  (day name)
df3["date"].dt.strftime("%a")          # Mon     (day abbr)                                                                                                                             
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# Rule of thumb:                                                                                                                                                                          
# - Use month_name() when you just need the month name — more readable
# - Use strftime() when you need custom date formatting

In [13]:
                                       
 
import pandas as pd                                                                                                                                                    
from ydata_profiling import ProfileReport
                                                                                                                                                                        
# load titanic from seaborn (no file needed)                                                                                                                           
import seaborn as sns
df = sns.load_dataset("titanic")                                                                                                                                       
                                                        
# profile = ProfileReport(df, title="Titanic EDA", explorative=True)
# profile.to_notebook_iframe()
   
# As percentages instead of counts                                                                                                                                                      
df['sex'].value_counts(normalize=True)
# male      0.647                                                                                                                                                                       
# female    0.353
                                                                                                                                                                                        
# Include NaN values
df['deck'].value_counts(dropna=False)
                                                                                                                                                                                        
# Sort by index instead of count
df['pclass'].value_counts(sort=False)                                                                                                                                                   
                                                                                                                                                                                        
  # Limit to top N
df['embarked'].value_counts().head(2)   
df.value_counts(['sex', 'pclass', 'survived'])     # Returns count for each combination of sex + pclass
df[['sex', 'pclass', 'survived']].value_counts()

sex     pclass  survived
male    3       0           300
female  1       1            91
male    2       0            91
        1       0            77
female  3       1            72
                0            72
        2       1            70
male    3       1            47
        1       1            45
        2       1            17
female  2       0             6
        1       0             3
Name: count, dtype: int64